# M3L4 E08 — LangGraph basico + Langfuse
### Modulo 3 · Lecture 4 · Construccion, pruebas y trazabilidad de agentes en produccion

---

## Que necesitas saber antes

| Modulo | Concepto | Por que lo necesitas aca |
|---|---|---|
| M3L4 E00-E03 | Trace, Span, MiniTracer | Langfuse hace lo mismo pero automatico y visual |
| M3L4 E01 | estructura trace -> spans -> generations | Langfuse usa exactamente esta misma jerarquia |
| M3L2 | LangChain: ChatOpenAI, chains | LangGraph es el siguiente nivel: grafos de estado |
| M3L3 | Agentes con estado, routing | LangGraph modela agentes como nodos de un grafo |
| Python | `TypedDict`, `Annotated` | Para definir el State del grafo con tipos |

Si no trabajaste con LangChain (M3L2), revisa los conceptos de `ChatOpenAI` y `messages` antes de empezar.

---

## Definiciones clave

| Concepto | Definicion simple | Como aparece en este notebook |
|---|---|---|
| **LangGraph** | Framework para construir agentes como grafos de estado | `StateGraph`, `START`, `END`, nodos y aristas |
| **State** | Estado compartido que fluye entre nodos del grafo | Clase `State(TypedDict)` con `messages: Annotated[list, add_messages]` |
| **Nodo** | Funcion que procesa el estado y devuelve una actualizacion | `chatbot_node(state)` que llama al LLM |
| **Arista** | Conexion entre nodos que define el flujo | `graph.add_edge(START, 'chatbot')`, `graph.add_edge('chatbot', END)` |
| **Langfuse CallbackHandler** | Interceptor que captura automaticamente traces de LangGraph | `CallbackHandler()` pasado en `config` |
| **Generation** | Sub-span de una llamada al LLM (modelo, tokens, latencia) | Langfuse lo captura automaticamente del `ChatOpenAI` |
| **Compilar** | Convertir la definicion del grafo en un ejecutable | `graph.compile()` |

---

## Como funciona la integracion

```
LangGraph graph
    |
    v
graph.invoke(..., config={'callbacks': [langfuse_handler]})
    |
    v
Langfuse CallbackHandler captura automaticamente:
    - Inputs/outputs de cada nodo
    - Llamadas al LLM (modelo, tokens, latencia)
    - Metadatos de la sesion
    |
    v
Langfuse UI -> Traces, spans, generations
```

### Jerarquia en Langfuse

```
Trace (toda la ejecucion del grafo)
  +-- Span (cada nodo del grafo)
        +-- Generation (cada llamada al LLM)
```

**Objetivo del ejercicio:** conectar un grafo simple de LangGraph con Langfuse para observar trazas en tiempo real.

## Paso 1 — Instalacion

| Libreria | Que hace | Por que la necesitamos |
|---|---|---|
| `langfuse` | SDK de Langfuse para tracing | `CallbackHandler` que captura traces automaticamente |
| `langchain` | Framework de aplicaciones LLM | Base para `StateGraph` y `ChatOpenAI` |
| `langchain-openai` | Integracion de OpenAI con LangChain | `ChatOpenAI` para llamar a GPT-4o-mini |
| `langgraph` | Framework de grafos de estado para agentes | `StateGraph`, `START`, `END`, nodos |
| `typing_extensions` | `TypedDict` para definir el State | Tipado del estado que fluye por el grafo |

```python
!pip install -q langfuse langchain langchain-openai langgraph
```

In [ ]:
!pip install -q langfuse langchain langchain-openai langgraph
print('Instalacion completa.')

## Paso 2 — Credenciales

> Consegui tus credenciales en [cloud.langfuse.com](https://cloud.langfuse.com) -> Settings -> API Keys

Necesitas 3 claves:

| Variable | Donde obtenerla |
|---|---|
| `LANGFUSE_PUBLIC_KEY` | Langfuse -> Settings -> API Keys -> Public Key |
| `LANGFUSE_SECRET_KEY` | Langfuse -> Settings -> API Keys -> Secret Key |
| `LANGFUSE_BASE_URL` | `https://cloud.langfuse.com` (EU) o `https://us.cloud.langfuse.com` (US) |
| `OPENAI_API_KEY` | [platform.openai.com/api-keys](https://platform.openai.com/api-keys) |

**Seguridad:** las credenciales se piden con `getpass()` para que no queden visibles en el notebook.

In [ ]:
import os
from getpass import getpass

os.environ['LANGFUSE_PUBLIC_KEY'] = getpass('Langfuse Public Key (pk-lf-...): ')
os.environ['LANGFUSE_SECRET_KEY'] = getpass('Langfuse Secret Key (sk-lf-...): ')
os.environ['LANGFUSE_BASE_URL']   = 'https://cloud.langfuse.com'
os.environ['OPENAI_API_KEY']      = getpass('OpenAI API Key: ')

print('Credenciales configuradas.')

## Paso 3 — LangGraph basico

### Explicacion de imports

| Import | Que hace |
|---|---|
| `from typing import Annotated` | Permite agregar metadatos a los tipos (como `add_messages`) |
| `from typing_extensions import TypedDict` | Define el State como un diccionario tipado |
| `from langgraph.graph import StateGraph, START, END` | Clases para construir el grafo y sus puntos de entrada/salida |
| `from langgraph.graph.message import add_messages` | Funcion que acumula mensajes en el state (append en vez de reemplazar) |
| `from langchain_core.messages import HumanMessage` | Mensaje del usuario en el formato de LangChain |
| `from langchain_openai import ChatOpenAI` | Cliente de OpenAI envuelto para LangChain |
| `from langfuse.langchain import CallbackHandler` | Handler que Langfuse usa para interceptar traces |

### El grafo mas simple posible

Un solo nodo que recibe un mensaje y responde:

```python
class State(TypedDict):
    messages: Annotated[list, add_messages]
    # State tiene una lista de mensajes
    # add_messages asegura que los nuevos mensajes se agreguen, no reemplacen
```

```python
graph = StateGraph(State)
graph.add_node('chatbot', chatbot_node)
graph.add_edge(START, 'chatbot')
graph.add_edge('chatbot', END)
graph = graph.compile()
```

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langfuse.langchain import CallbackHandler

print('Imports OK.')

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
print('State y LLM listos.')

## TODO — Completar el grafo

Completa los pasos para construir, compilar y ejecutar el grafo.

### chatbot_node(state)

| Parametro | Tipo | Que es |
|---|---|---|
| `state` | `State` | Diccionario con `messages: list` |
| **Retorna** | `dict` | `{'messages': [respuesta_del_llm]}` |

El nodo debe:
1. Tomar los mensajes del state
2. Llamar al LLM
3. Devolver la respuesta como actualizacion del state

In [ ]:
def chatbot_node(state: State) -> dict:
    """Nodo que llama al LLM con los mensajes actuales."""
    # TODO: invocar el LLM con state['messages'] y retornar {'messages': [response]}
    pass

print('Nodo definido.')

In [ ]:
# TODO: construir el grafo
# 1. Crear StateGraph con State
# 2. Agregar nodo 'chatbot' con chatbot_node
# 3. Agregar arista START -> 'chatbot' (set_entry_point)
# 4. Agregar arista 'chatbot' -> END (set_finish_point)
# 5. Compilar

graph = None  # reemplazar con la compilacion
print('Grafo compilado.')

## Paso 4 — Ejecutar con Langfuse

El truco es pasar el `CallbackHandler` en el `config`.

```python
langfuse_handler = CallbackHandler()

result = graph.invoke(
    {'messages': [HumanMessage(content='...')]},
    config={'callbacks': [langfuse_handler]}
)
```

Langfuse captura automaticamente:
- El input (`HumanMessage`)
- El output (respuesta del LLM)
- Los tokens usados
- La latencia
- El modelo

Sin el `CallbackHandler`, el grafo funciona igual pero no queda registrado en Langfuse.

In [ ]:
langfuse_handler = CallbackHandler()

# TODO: invocar el grafo con:
# - input: {'messages': [HumanMessage(content='Explicame que es tracing en IA en 2 oraciones')]}
# - config: {'callbacks': [langfuse_handler]}
result = None  # reemplazar

if result:
    print('Respuesta:', result['messages'][-1].content)

## Visualizar el grafo

LangGraph puede mostrar la estructura del grafo como diagrama Mermaid.

In [ ]:
if graph:
    print(graph.get_graph().draw_mermaid())

## Que ver en Langfuse

Despues de ejecutar la celda anterior:

1. Abri [cloud.langfuse.com](https://cloud.langfuse.com)
2. Anda a **Traces**
3. Busca la trace recien creada
4. Revisa:

| Campo | Que muestra |
|---|---|
| **Input** | El mensaje enviado al grafo |
| **Output** | La respuesta del LLM |
| **Modelo** | `gpt-4o-mini` |
| **Tokens** | Input + output tokens usados |
| **Duracion** | Latencia total de la ejecucion |
| **Spans** | Cada nodo del grafo (aca solo `chatbot`) |
| **Generations** | La llamada al LLM dentro del nodo |

In [ ]:
assert graph is not None, 'El grafo no fue compilado'
assert result is not None, 'No se obtuvo resultado'
assert len(result['messages']) > 0, 'No hay mensajes en el resultado'
print('Checks E08 OK')

## Errores comunes

| Error | Causa | Como detectarlo |
|---|---|---|
| `graph` es `None` | No llamar `graph.compile()` | El assert `graph is not None` falla |
| No aparece en Langfuse | Olvidar pasar `config={'callbacks': [langfuse_handler]}` | La ejecucion funciona pero no hay trace en Langfuse |
| `result` es `None` | No invocar `graph.invoke()` | `result['messages']` da error |
| `Invalid API Key` | Credenciales incorrectas o vencidas | Langfuse lanza error de autenticacion |
| `ModuleNotFoundError` | No instalar las librerias primero | Ejecutar celda de instalacion |
| `add_messages` no funciona | No importar `Annotated` de `typing` | Error de tipo al crear el State |

## Sintesis

### Que construiste

| Componente | Descripcion |
|---|---|
| `State(TypedDict)` | Estado del grafo con lista de mensajes |
| `chatbot_node()` | Nodo que llama al LLM y devuelve respuesta |
| `StateGraph` | Grafo con un nodo, entrada y salida |
| `graph.compile()` | Grafo listo para ejecutar |
| `CallbackHandler()` | Interceptor de Langfuse para tracing automatico |

### Diferencia con MiniTracer (E01-E02)

| Aspecto | MiniTracer | Langfuse CallbackHandler |
|---|---|---|
| Donde se crea | Manual: `tracer.start_trace()` | Automatico: al pasar el handler en config |
| Spans | Manual: `tracer.add_span()` | Automatico: cada nodo es un span |
| Generations | No disponible | Automatico: cada llamada al LLM es una generation |
| Visualizacion | Diccionario en Python | UI web en cloud.langfuse.com |
| Persistencia | En memoria | Guardado en Langfuse |

### Relacion con otros ejercicios

| Ejercicio | Conexion con E08 |
|---|---|
| **E09** | LangGraph router: grafo con decision condicional + Langfuse |
| **E10** | Multi-agente supervisor: grafo con routing entre agentes + Langfuse |
| **E11** | Golden dataset scores sobre grafos de LangGraph |